In [0]:
rawdf1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/custsmodified",header=False,inferSchema=True).toDF("id","firstname","lastname","age","profession")
rawdf1.show(20,False)


In [0]:
rawdf1.printSchema()
print(rawdf1.columns)
print(rawdf1.dtypes)
for i in rawdf1.dtypes:
    if i[1]=='string':
        print(i[0])
print(rawdf1.schema)

In [0]:
print("actual count of the data",rawdf1.count())
print("de-duplicated record (all columns) count",rawdf1.distinct().count())
print("de-duplicated record (all columns) count",rawdf1.dropDuplicates().count())
print("de-duplicated given cid column count",rawdf1.dropDuplicates(['id']).count())
display(rawdf1.describe())
display(rawdf1.summary())

1. **Structuring - Combining Data + Schema Evolution/Merging**

In [0]:
strt1="id string, firstname string, lastname string, age string, profession string"
rawdf1=spark.read.schema(strt1).csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/"],recursiveFileLookup=True,pathGlobFilter="custsmodified_N*")
display(rawdf1)
strt2="id string, firstname string, age string, profession string, lastname string"
rawdf2=spark.read.schema(strt2).csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/"],recursiveFileLookup=True,pathGlobFilter="custsmodified_T*")
display(rawdf2)
#rawdf_merged=rawdf1.union(rawdf2)#Use union only if the dataframes are having same columns in the same order with same datatype..
#display(rawdf_merged)
rawdf_merged=rawdf1.unionByName(rawdf2,allowMissingColumns=True)
display(rawdf_merged)

**2. Validation, Cleansing, Scrubbing - Cleansing (removal of unwanted datasets), Scrubbing (convert raw to tidy)**

In [0]:
from pyspark.sql.types import *
struttype1=StructType([StructField('id', IntegerType(), True), StructField('firstname', StringType(), True), StructField('lastname', StringType(), True), StructField('age', ShortType(), True), StructField('profession', StringType(), True)])
#method1 - permissive with all rows with respective nulls
cleandf1=spark.read.schema(struttype1).csv(path="/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/custsmodified",mode='permissive')
print("after keeping nulls on the wrong data format",cleandf1.count())#all rows count
display(cleandf1.show(20))#We are making nulls where ever data format mismatch is there (cutting down mud portition from potato)
#method2 - drop malformed rows
cleandf2=spark.read.schema(struttype1).csv(path="/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/custsmodified", mode='dropMalformed')
print("dropped malformed rows", (cleandf2.count()))
print("after cleaning wrong data (type mismatch, column number mismatch)",len(cleandf2.collect()))
display(cleandf2.show(20))

In [0]:
#method3 best methodology of applying active data munging
#Validation by doing cleansing (not at the time of creating Dataframe, rather we will clean and scrub subsequently)...
from pyspark.sql.types import *
struttype1=StructType([StructField('id', StringType(), True), StructField('firstname', StringType(), True), StructField('lastname', StringType(), True), StructField('age', StringType(), True), StructField('profession', StringType(), True)])
#method1 - permissive with all rows with respective nulls
rawdf1=spark.read.schema(struttype1).csv(path="/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/custsmodified",mode='permissive')
print("allow all data showing the real values",rawdf1.count())#all rows count
display(rawdf1.show(20))#We are making nulls where ever data format mismatch is there (cutting down mud portition from potato)

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,ShortType,IntegerType
struttype1=StructType([StructField('id', IntegerType(), True), StructField('firstname', StringType(), True), StructField('lastname', StringType(), True), StructField('age', ShortType(), True), StructField('profession', StringType(), True),StructField("corruptedrows",StringType())])
#method1 - permissive with all rows with respective nulls
cleandf1=spark.read.schema(struttype1).csv(path="/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/custsmodified",mode='permissive',columnNameOfCorruptRecord="corruptedrows")
display(cleandf1.show(20))

#Create a reject dataset
rejectdf1=cleandf1.where("corruptedrows is not null")
display(rejectdf1)
rejectdf1.write.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/reject",mode="overwrite",header=True)
#dbutils.fs.ls("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/directory_dropme/reject")
retained_df=cleandf1.where("corruptedrows is null")
display(retained_df.show(20))   

In [0]:
print("Overall rows in the source data is ",len(cleandf1.collect()))
print("Rejected rows in the source data is ",len(rejectdf1.collect()))
print("Clean rows in the source data is ",len(retained_df.collect()))

In [0]:
cleanseddf=rawdf1.na.drop(how="any")
print("count of cleanseddf", cleanseddf.count())
print("any one row in the raw df with age null")
display(rawdf1.where("age is null"))
print("any one row in the cleansed df with age null")
display(cleanseddf.where("age is null"))#any one column contains null will be cleaned
cleanseddf=rawdf1.na.drop(how="any",subset=["id","age"])#If we need CDE without nulls (Critical Data Elements/Significant columns) columns
print("any one row in the cleansed df with id or age null")
display(cleanseddf.take(20))
cleanseddf=rawdf1.na.drop(how="all",subset=["firstname","lastname"])#4000004,Gretchen,,66,
print("any one row in the cleansed df with firstname and lastname is null")
print("Total rows without firstname and lastname with null values", cleanseddf.count())
display(cleanseddf.show(20))#We are taking this DF further for munging..

In [0]:
scrubbeddf1=cleanseddf.na.fill('not provided',subset=["lastname","profession"])#fill will help us replace nulls with some value
display(scrubbeddf1)
find_replace_values_dict1={'Pilot':'Captain','Actor':'Celeberity'}
find_replace_values_dict2={'not provided':'NA'}
scrubbeddf2=scrubbeddf1.na.replace(find_replace_values_dict1,subset=["profession"])
display(scrubbeddf2)
scrubbeddf3=scrubbeddf2.na.replace(find_replace_values_dict2,subset=["lastname"])
display(scrubbeddf3)

3. Standardization and Replacement / Deletion of Data to make it in a usable format

Standardization1 - Column Enrichment (Addition of columns)

In [0]:
from pyspark.sql.functions import lit,initcap,col
#withColumn("stringcolumnname to add in the df",lit('hardcoded')/initcap(col("colname")))
standarddf1=scrubbeddf3.withColumn("sourcesystem",lit("Retail"))#SparkSQL - DSL(FBP)
display(standarddf1.limit(20))

Standardization2 - Column Uniformity

In [0]:
from pyspark.sql.functions import upper
#Basic Exploration/analysis of the profession column for identifying uniformity challenges
#standarddf1.createOrReplaceTempView("sqlview")
#display(spark.sql("select profession,count(*) from sqlview group by profession order by profession"))#SQL
#display(standarddf1.groupBy("profession").count())#DSL

#Standardization2 - column uniformity
standarddf2=standarddf1.withColumn("profession",initcap("profession"))#inicap or any other string function with columnOr name can accept either column or string type provided if the string is a column name for eg. profession/age/sourcesystem.
display(standarddf2.limit(20))
#display(standarddf2.groupBy("profession").count())#DSL

Standardization3 - Format Standardization

In [0]:
#Did analysis to understand the format issues in our id and age columns
standarddf2.where("id like 't%'").show()
standarddf2.where("id rlike '[a-zA-Z]'").show()#rlike is regular expression like function that help us identify any string data in our DF column
standarddf2.where("age rlike '[^0-9]'").show()#checking for any non number values in age column
display(standarddf2.where("id in ('ten')"))
standarddf2.show()

In [0]:
from pyspark.sql.functions import regexp_replace,replace
replaceval = {'one':'1','two':'2','three':'3', 'four':'4', 'five':'5', 'six':'6', 'seven':'7', 'eight':'8', 'nine':'9', 'ten':'10'}
standarddf3=standarddf2.na.replace(replaceval,["id"])
#standarddf3=standarddf2.withColumn("id",replace("id",lit('ten'),"10"))
standarddf3=standarddf3.withColumn("age",regexp_replace("age",'-',""))
display(standarddf3)

Standardization4 - Data Type Standardization

In [0]:
standarddf3.printSchema()#Still id and age are string type, though it contains int data
standarddf4=standarddf3.withColumn("id",col("id").cast("long"))
standarddf4=standarddf4.withColumn("age",col("age").cast("short"))
standarddf4.printSchema()
display(standarddf4)

Standardization5 - Naming Standardization

In [0]:
standarddf5=standarddf4.withColumnRenamed("id","custid")
standarddf5=standarddf4.withColumnsRenamed({"id":"custid","sourcesystem":"srcsystem"})
display(standarddf5)

Standardization6 - Reorder Standadization

In [0]:
standarddf6=standarddf5.select("custid", "age", "firstname","lastname","profession","srcsystem")
display(standarddf6)
from pyspark.sql.functions import col, when, concat_ws

standarddf7 = standarddf6.withColumn(
    "_corrupt_record",
    when(col("custid").isNull(), "custid is null")
    .when(col("age").isNull(), "age is null")
    .otherwise(None)
)

display(standarddf7)
#mungeddf=standarddf6
#display(mungeddf.take(10))

**DeDuplication**

In [0]:
display(standarddf6.where("custid in ('4000001')"))#before row level dedup
dedupdf1=standarddf6.distinct()#It will remove the row level duplicates
display(dedupdf1.where("custid in ('4000001')"))

print("non prioritized deduplication, just remove the duplicates retaining only the first row")
display(dedupdf1.coalesce(1).where("custid in ('4000003')"))#before col level dedup
dedupdf2=dedupdf1.coalesce(1).dropDuplicates(subset=["custid"])#It will remove the column level duplicates (retaining the first row in the dataframe)
display(dedupdf2.where("custid in ('4000003')"))
dedupdf2=dedupdf1.coalesce(1).orderBy(["custid","age"],ascending=[True,False]).dropDuplicates(subset=["custid"])
display(dedupdf2.where("custid in ('4000003')"))
display(dedupdf2)




In [0]:
mungeddf = dedupdf2
display(mungeddf.take(20))

**2. Data Enrichment - Detailing of data**

1.Adding of columns

In [0]:
derived_datadt='25/30/12'
print(f"hello '{derived_datadt}'")

In [0]:
from pyspark.sql.functions import lit,current_date
original_filename='custsmodified_25/30/12.csv'#We are deriving this date from the filename provided by the source custsmodified_25/30/12.csv
derived_datadt=original_filename.split('_')[1].split('.')[0]

enrichdf1=mungeddf.withColumn("datadt",lit('25/30/12')).withColumn("loaddt",current_date())
enrichdf1.printSchema()
display(enrichdf1)

Deriving of columns

In [0]:
from pyspark.sql.functions import *
enrichdf2=enrichdf1.select("*",substring("profession",1,1).alias("professionflag"))
display(enrichdf2.take(20))

Renaming of columns

In [0]:
enrichdf3=enrichdf2.withColumnRenamed("srcsystem","sourcename")#Best function to rename the column(s)
display(enrichdf3.take(20))
enrichdf3=enrichdf2.withColumnsRenamed({"srcsystem":"sourcename","professionflag":"profflag"})
display(enrichdf3.take(20))

Modify/replace (withColumn, select/selectExpr)

In [0]:
enrichdf4=enrichdf3.withColumn("profession",concat("profession",lit('-'),"profflag"))#This will modify/enrich the profession column with sourcename
display(enrichdf4.take(20))

Remove/Eliminate (drop,select,selectExpr)

In [0]:
enrichdf5=enrichdf4.drop("profflag")#right function to use from dropping
display(enrichdf5.take(20))

In [0]:
name='irfan'
sqlexpression=f"'{name}' as owner"
print(sqlexpression)
mungeddf.selectExpr("*",sqlexpression).display()

b. Splitting & Merging/Melting of Columns

In [0]:
splitdf=enrichdf5.withColumn("profflag",split("profession",'-'))
splitdf2=splitdf.withColumn("profession",col("profflag")[0])
display(splitdf2.take(20))
splitdf2=splitdf2.withColumn("shortprof",upper(substring(col("profession"),1,3))).drop("profflag")
display(splitdf2.take(20))

mergeddf=splitdf2.select(col("custid"),"age",concat_ws(" ",col("firstname"),col("lastname")).alias("fullname"),"profession","sourcename","datadt","loaddt","shortprof")#usage of select will help us avoid chaining of withColumn,drop,select
display(mergeddf.limit(20))

In [0]:
mergeddf.printSchema()

In [0]:
cleandf = mergeddf.filter((col("custid").isNotNull()) & (col("custid") != 10) & (col("age").isNotNull ()))
display(cleandf.limit(20))